<div style="background-color:#000047; padding: 30px; border-radius: 10px; color: white; text-align: center;">
    <img src='Figures/alinco_white_text.png' style="height: 100px; margin-bottom: 10px;"/>
    <h1> Módulo 1: Crear una Red Neuronal Convolucional con TensorFlow</h1>
    <h3>Aprendizaje Automático Avanzado 2026</h3>
</div>

## Importar las librerías necesarias

En este notebook construiremos un clasificador de imágenes **binario** (caballo vs. humano) con una **red neuronal convolucional (CNN)** en TensorFlow 2 / Keras. Usaremos APIs modernas: `image_dataset_from_directory` para los datos, capas de **aumento de datos** y **normalización** dentro del modelo, y **BatchNormalization** + **Dropout** para regularizar.


In [ ]:
import os

# Evita el conflicto de múltiples runtimes OpenMP (MKL + libgomp) en Windows
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import zipfile
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import tensorflow as tf
from tensorflow.keras import layers

tf.random.set_seed(42)
print("TensorFlow:", tf.__version__)

## Descargar y preparar el dataset (Horses or Humans)

Usaremos el dataset **Horses or Humans**: imágenes de $300\times300$ píxeles a color de caballos y humanos generados por computadora. Construiremos un clasificador que, dada una imagen, determine si contiene un **caballo** o un **humano**.

La siguiente celda **descarga** los conjuntos de entrenamiento y validación (solo la primera vez) y los descomprime en una carpeta local `data/`. Cada conjunto contiene subcarpetas `horses/` y `humans/`; más adelante, `image_dataset_from_directory` **etiquetará automáticamente** las imágenes a partir del nombre de la subcarpeta.


In [ ]:
# Descargar y descomprimir el dataset en una carpeta local 'data/' (multiplataforma)
os.makedirs('data', exist_ok=True)

datasets = {
    'horse-or-human':            'https://storage.googleapis.com/learning-datasets/horse-or-human.zip',
    'validation-horse-or-human': 'https://storage.googleapis.com/learning-datasets/validation-horse-or-human.zip',
}

for nombre, url in datasets.items():
    out_dir = os.path.join('data', nombre)
    if not os.path.isdir(out_dir):
        zip_path = os.path.join('data', nombre + '.zip')
        if not os.path.exists(zip_path):
            print(f'Descargando {nombre} ...')
            urllib.request.urlretrieve(url, zip_path)
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(out_dir)
        print(f'  -> extraído en {out_dir}')
    else:
        print(f'{nombre} ya está disponible.')


In [ ]:
train_dir = os.path.join('data', 'horse-or-human')
val_dir   = os.path.join('data', 'validation-horse-or-human')

## Explorar los datos

El dataset está organizado en carpetas: el conjunto de **entrenamiento** contiene las subcarpetas `horses/` y `humans/`, y lo mismo el de **validación**. 

Una particularidad importante: **no etiquetamos las imágenes manualmente**. A diferencia de MNIST o Fashion-MNIST (donde las etiquetas venían dadas), aquí la etiqueta se deduce del **nombre de la subcarpeta** en la que está cada imagen. La utilidad `image_dataset_from_directory` se encarga de leer las imágenes y asignarles la clase automáticamente.

Definamos las rutas a cada subdirectorio:


In [ ]:
# Rutas a los subdirectorios de cada clase
train_horse_dir = os.path.join(train_dir, 'horses')
train_human_dir = os.path.join(train_dir, 'humans')
validation_horse_dir = os.path.join(val_dir, 'horses')
validation_human_dir = os.path.join(val_dir, 'humans')

In [ ]:
validation_human_dir

Ahora, veamos cómo se ven los nombres de los archivos en los directorios de entrenamiento de caballos y humanos:

In [ ]:
train_horse_names = os.listdir(train_horse_dir)
print(train_horse_names[:10])

In [ ]:
train_humans_names = os.listdir(train_human_dir)
print(train_humans_names[:10])

Averigüemos el número total de imágenes de caballos y humanos en los directorios:

In [ ]:
len(train_humans_names), len(train_horse_names)

Ahora echemos un vistazo a algunas imágenes para tener una mejor idea de cómo se ven. Primero, configure los parámetros de matplot:

In [ ]:
nrows= 4
ncols = 4
pic_index=0

Now, display a batch of 8 horse and 8 human pictures. You can rerun the cell to see a fresh batch each time:

In [ ]:
fig = plt.gcf()
fig.set_size_inches(ncols*4, nrows*4)

pic_index +=8
next_horse_pix = [os.path.join(train_horse_dir, fname) for fname in train_horse_names[pic_index -8 : pic_index]]
next_human_pix = [os.path.join(train_human_dir, fname) for fname in train_humans_names[pic_index -8 : pic_index]]

for i, img_path in enumerate(next_horse_pix+next_human_pix):
    sp=plt.subplot(nrows,ncols, i+1)
    sp.axis('off')
    
    img = mpimg.imread(img_path)
    plt.imshow(img)
    
plt.show()


In [ ]:
img.shape

## Preparar los datos con `image_dataset_from_directory`

En lugar del antiguo `ImageDataGenerator` (obsoleto), usamos **`tf.keras.utils.image_dataset_from_directory`**, que crea un `tf.data.Dataset` eficiente leyendo las imágenes de las carpetas y etiquetándolas por subcarpeta.

- Redimensionamos a $150\times150$ (más rápido que $300\times300$ y suficiente para este problema).
- `label_mode='binary'` porque son 2 clases.
- Aplicamos `cache()` y `prefetch()` para acelerar el entrenamiento.

> La **normalización** de los píxeles a $[0,1]$ la haremos con una capa `Rescaling` **dentro del modelo**, así el mismo preprocesamiento se aplica automáticamente en entrenamiento e inferencia.


In [ ]:
IMG_SIZE = (150, 150)
BATCH_SIZE = 32

# image_dataset_from_directory



In [ ]:
# Optimización del pipeline: cache + prefetch
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

## Construir la red neuronal convolucional

### Definir el modelo

Nuestra CNN apila varios bloques de **`Conv2D` → `BatchNormalization` → `MaxPooling2D`**, que extraen características cada vez más complejas mientras reducen el tamaño espacial. Mejoras respecto al modelo original:

- **Aumento de datos** (`RandomFlip`, `RandomRotation`, `RandomZoom`) al inicio: genera variaciones de las imágenes en cada época para **reducir el sobreajuste** (solo actúa durante el entrenamiento).
- **`Rescaling(1./255)`**: normaliza los píxeles a $[0,1]$ dentro del modelo.
- **`BatchNormalization`**: estabiliza y acelera el entrenamiento.
- **`Dropout`**: regularización adicional antes de la capa densa.

Como es un problema de clasificación **binaria**, la red termina con **una** neurona y activación **sigmoide**, cuya salida es la probabilidad (entre 0 y 1) de pertenecer a la clase 1.


In [ ]:
# Bloque de aumento de datos (solo se aplica durante el entrenamiento)
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name='data_augmentation')

# Crear la arquitectura de la red, model


In [ ]:
# resumen de la arquitectura


La columna "forma de salida" muestra cómo evoluciona el tamaño de su mapa de características en cada capa sucesiva. Las capas de convolución reducen un poco el tamaño de los mapas de características debido al relleno, y cada capa de agrupación reduce a la mitad las dimensiones.

A continuación, configuraremos las especificaciones para el entrenamiento del modelo. Entrenaremos nuestro modelo con la pérdida de entropía cruzada binaria, porque es un problema de clasificación binaria y nuestra activación final es un sigmoide. (Para refrescar las métricas de pérdida, consulte el Curso intensivo de aprendizaje automático). Usaremos el optimizador rmsprop con una tasa de aprendizaje de 0,001. Durante el entrenamiento, querremos monitorear la precisión de la clasificación.

NOTA: En este caso, usar el algoritmo de optimización RMSprop es preferible al descenso de gradiente estocástico (SGD), porque RMSprop automatiza el ajuste de la tasa de aprendizaje para nosotros. (Otros optimizadores, como Adam y Adagrad, también adaptan automáticamente la tasa de aprendizaje durante el entrenamiento y funcionarían igual de bien aquí).

### Compilar el modelo



In [ ]:
# RMSprop adapta la tasa de aprendizaje automáticamente (Adam funcionaría igual de bien)


### Entrenar el modelo

Entrenamos durante 15 épocas usando los `tf.data.Dataset` que preparamos. Guardamos el `history` para graficar las curvas de aprendizaje. En CPU puede tardar varios minutos.

In [ ]:
# Guardar el historial del loss


### Curvas de aprendizaje

Graficamos la pérdida y la precisión de entrenamiento frente a las de validación para diagnosticar el aprendizaje (y detectar sobreajuste).

In [ ]:
h = history.history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(h['loss'], marker='o', label='entrenamiento')
ax1.plot(h['val_loss'], marker='o', label='validación')
ax1.set_title('Pérdida'); ax1.set_xlabel('época'); ax1.set_ylabel('loss')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(h['accuracy'], marker='o', label='entrenamiento')
ax2.plot(h['val_accuracy'], marker='o', label='validación')
ax2.set_title('Precisión'); ax2.set_xlabel('época'); ax2.set_ylabel('accuracy')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Evaluar el modelo y hacer predicciones

Primero medimos la precisión sobre el conjunto de **validación** completo. Luego mostramos predicciones sobre imágenes concretas: recuerda que la salida es una probabilidad; si es **mayor que 0.5** la clase predicha es la 1 (`humans`), y si no, la clase 0 (`horses`).

In [ ]:
# Precisión sobre todo el conjunto de validación


In [ ]:
# Predicciones sobre un lote de validación (verde = acierto, rojo = error)
images, labels = next(iter(val_ds))
probs = model.predict(images, verbose=0).ravel()
preds = (probs > 0.5).astype(int)
reales = labels.numpy().ravel().astype(int)

plt.figure(figsize=(12, 7))
for i in range(min(12, len(images))):
    plt.subplot(3, 4, i + 1)
    plt.imshow(images[i].numpy().astype('uint8'))
    correcto = preds[i] == reales[i]
    color = 'green' if correcto else 'red'
    plt.title(f'{class_names[preds[i]]} ({probs[i]:.2f})', color=color, fontsize=10)
    plt.axis('off')
plt.tight_layout()
plt.show()